<a href="https://colab.research.google.com/github/vaalkate/ITMO_High_Performance_Graph_Analysis/blob/main/%D0%97%D0%B0%D0%B4%D0%B0%D1%87%D0%B0_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задача 6. BFS on GPU


### Задание 1
Используя OpenCL или CUDA реализовать алгоритм обхода в ширину ориентированного графа на GPU. Реализация не обязательно должна использовать операции линейной алгебры, можно взять классический алгоритм BFS.

In [ ]:
import numpy as np
from numba import cuda
import math
import time
import os
import glob

In [ ]:
@cuda.jit
def bfs_kernel(row_ptr, col_ind, frontier, next_frontier, visited, dist, level):
    u = cuda.grid(1)

    if u >= frontier.size:
        return

    if frontier[u] == 1:
        start = row_ptr[u]
        end = row_ptr[u + 1]

        for i in range(start, end):
            v = col_ind[i]

            # атомарная проверка
            old = cuda.atomic.exch(visited, v, 1)

            if old == 0:
                dist[v] = level + 1
                next_frontier[v] = 1


In [ ]:
# BFS GPU

def bfs_gpu(n, row_ptr, col_ind, source):
    # GPU массивы
    d_row_ptr = cuda.to_device(row_ptr)
    d_col_ind = cuda.to_device(col_ind)

    frontier = np.zeros(n, dtype=np.int32)
    next_frontier = np.zeros(n, dtype=np.int32)
    visited = np.zeros(n, dtype=np.int32)
    dist = np.full(n, -1, dtype=np.int32)

    # стартовая вершина
    frontier[source] = 1
    visited[source] = 1
    dist[source] = 0

    d_frontier = cuda.to_device(frontier)
    d_next_frontier = cuda.to_device(next_frontier)
    d_visited = cuda.to_device(visited)
    d_dist = cuda.to_device(dist)

    threads_per_block = 256
    blocks = math.ceil(n / threads_per_block)

    level = 0

    while True:
        # обнуляем следующий фронт
        d_next_frontier[:] = 0

        bfs_kernel[blocks, threads_per_block](
            d_row_ptr,
            d_col_ind,
            d_frontier,
            d_next_frontier,
            d_visited,
            d_dist,
            level,
        )

        # проверяем есть ли новые вершины
        next_frontier = d_next_frontier.copy_to_host()
        if next_frontier.sum() == 0:
            break


        d_frontier = cuda.to_device(next_frontier)
        level += 1

    return d_dist.copy_to_host()

In [ ]:
# Вспомогательная функция

def build_csr(src, dst, n):
    """
    Строит CSR представление графа
    """
    edges = list(zip(src, dst))
    edges.sort()

    row_ptr = np.zeros(n + 1, dtype=np.int32)
    col_ind = []

    current = 0
    for u, v in edges:
        while current < u:
            row_ptr[current + 1] = len(col_ind)
            current += 1
        col_ind.append(v)

    while current < n:
        row_ptr[current + 1] = len(col_ind)
        current += 1

    return row_ptr, np.array(col_ind, dtype=np.int32)


In [ ]:
# CPU BFS

def bfs_cpu(n, row_ptr, col_ind, source):
    dist = [-1] * n
    dist[source] = 0

    queue = [source]

    while queue:
        u = queue.pop(0)
        for i in range(row_ptr[u], row_ptr[u + 1]):
            v = col_ind[i]
            if dist[v] == -1:
                dist[v] = dist[u] + 1
                queue.append(v)

    return np.array(dist)

In [ ]:
# Тест

def main():

    n = 6
    src = np.array([0, 0, 1, 2, 3, 4])
    dst = np.array([1, 2, 3, 3, 4, 5])

    row_ptr, col_ind = build_csr(src, dst, n)

    source = 0

    print("Запуск BFS на CPU")
    t0 = time.time()
    dist_cpu = bfs_cpu(n, row_ptr, col_ind, source)
    t_cpu = time.time() - t0

    print("Запуск BFS на GPU")
    t0 = time.time()
    dist_gpu = bfs_gpu(n, row_ptr, col_ind, source)
    t_gpu = time.time() - t0

    print("\nРезультаты:")
    print("CPU dist:", dist_cpu)
    print("GPU dist:", dist_gpu)

    print(f"\nВремя CPU: {t_cpu:.6f}с")
    print(f"Время GPU: {t_gpu:.6f}с")

    print("\nПроверка корректности:",
          np.all(dist_cpu == dist_gpu))


if __name__ == "__main__":
    main()

Запуск BFS на CPU
Запуск BFS на GPU

Результаты:
CPU dist: [0 1 1 2 3 4]
GPU dist: [0 1 1 2 3 4]

Время CPU: 0.000032с
Время GPU: 0.182763с

Проверка корректности: True


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


**Вывод:** реализация алгоритма обхода в ширину на GPU функционирует корректно. Расстояния до вершин, вычисленные на CPU и GPU, полностью совпадают, что подтверждает правильность реализации и соответствие классическому алгоритму BFS.

Различие во времени выполнения объясняется размером тестового графа. Для небольшого графа накладные расходы, связанные с передачей данных на видеокарту и запуском CUDA-ядра, существенно превышают время самих вычислений, поэтому GPU-версия оказывается значительно медленнее CPU. Это является ожидаемым поведением, так как преимущества параллельных вычислений проявляются только на графах большого разме

### Задание 2
Провести экспериментальное исследование полученной реализации и сравнить её с реализацией этого же алгоритма на CPU. Графы брать разного размера, например с сайта SuiteSparse Matrix Collection. На некоторых графах добиться прироста производительности при использовании GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def run_bfs_experiment(filepath):
    print(f"\nЗагрузка графа: {os.path.basename(filepath)}")

    src, dst, n = load_graph(filepath)
    m = len(src)

    print(f"  Вершин: {n:,}   Рёбер: {m:,}   Плотность: {m/(n*n):.2e}")

    row_ptr, col_ind = build_csr(src, dst, n)
    source = 0

    # CPU
    print("CPU BFS")
    t0 = time.perf_counter()
    dist_cpu = bfs_cpu(n, row_ptr, col_ind, source)
    t_cpu = time.perf_counter() - t0

    # GPU
    print("GPU BFS")
    t0 = time.perf_counter()
    dist_gpu = bfs_gpu(n, row_ptr, col_ind, source)
    t_gpu = time.perf_counter() - t0

    correct = np.all(dist_cpu == dist_gpu)

    speedup = t_cpu / t_gpu if t_gpu > 0 else float("inf")

    print("\nРезультаты:")
    print(f"  CPU: {t_cpu:.4f}с")
    print(f"  GPU: {t_gpu:.4f}с")
    print(f"  Ускорение: {speedup:.2f}x")
    print(f"  Корректность: {correct}")

    print("-" * 60)

    return n, t_cpu, t_gpu, speedup


In [ ]:
def main():
    folder = "/content/drive/MyDrive/Colab Notebooks/Графы/Matrix Market Big"

    # берём ВСЕ графы
    files = sorted(
        glob.glob(os.path.join(folder, "*.mtx")) +
        glob.glob(os.path.join(folder, "*.txt"))
    )

    if not files:
        print("Файлы не найдены")
        return

    results = []

    for f in files:
        try:
            res = run_bfs_experiment(f)
            results.append((os.path.basename(f), *res))
        except Exception as e:
            print(f"Ошибка на графе {f}: {e}")

    print("\nИтоговая таблица")
    print("-" * 60)
    print("Граф | Вершины | CPU | GPU | Speedup")

    for name, n, t_cpu, t_gpu, sp in results:
        print(f"{name} | {n:,} | {t_cpu:.3f}s | {t_gpu:.3f}s | {sp:.2f}x")


if __name__ == "__main__":
    main()


Загрузка графа: can_838.mtx
  Вершин: 838   Рёбер: 10,010   Плотность: 1.43e-02
CPU BFS
GPU BFS

Результаты:
  CPU: 0.0021с
  GPU: 0.0099с
  Ускорение: 0.21x
  Корректность: True
------------------------------------------------------------

Загрузка графа: dwt_992.mtx
  Вершин: 992   Рёбер: 16,744   Плотность: 1.70e-02
CPU BFS
GPU BFS

Результаты:
  CPU: 0.0033с
  GPU: 0.0284с
  Ускорение: 0.12x
  Корректность: True
------------------------------------------------------------

Загрузка графа: web-Google.mtx
  Вершин: 916,428   Рёбер: 5,105,039   Плотность: 6.08e-06
CPU BFS
GPU BFS

Результаты:
  CPU: 11.4137с
  GPU: 0.1242с
  Ускорение: 91.89x
  Корректность: True
------------------------------------------------------------

Загрузка графа: web-Stanford.mtx
  Вершин: 281,903   Рёбер: 2,312,497   Плотность: 2.91e-05
CPU BFS
GPU BFS

Результаты:
  CPU: 0.0166с
  GPU: 0.0461с
  Ускорение: 0.36x
  Корректность: True
------------------------------------------------------------

Итоговая та

**Вывод:** по результатам экспериментального исследования можно сделать вывод, что эффективность реализации BFS на GPU существенно зависит от размера и структуры графа. На малых графах, таких как can_838 и dwt_992, GPU не демонстрирует ускорения по сравнению с CPU и уступает ему по времени выполнения.

На графе web-Stanford также не наблюдается ускорения, и GPU остаётся медленнее CPU, что объясняется особенностями структуры графа и недостаточной загрузкой вычислительных блоков в процессе обхода в ширину. BFS в таких условиях формирует относительно узкие фронтиры, что ограничивает степень параллелизма.

В то же время на графе web-Google наблюдается значительное ускорение GPU-реализации, достигающее почти 92-кратного выигрыша по времени выполнения. Это связано с большим количеством вершин и рёбер, а также с более благоприятной структурой графа, при которой фронтиры обхода оказываются достаточно широкими для эффективной загрузки GPU. В этом случае параллельная обработка уровней BFS позволяет полностью раскрыть преимущества архитектуры графического процессора.

Таким образом, эксперимент подтверждает, что GPU-ускорение BFS проявляется только на достаточно крупных и структурно подходящих графах, тогда как на малых и средних графах CPU остаётся более эффективным решением.